In [4]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_API_KEY"] = "dummy"

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Configuración del modelo (igual al chat con ia.)
token = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.github.ai/inference"
model_name = "openai/gpt-4.1"

llm = ChatOpenAI(
    base_url=endpoint,
    api_key=token,
    model=model_name,
    temperature=0.1,
    streaming=True
)

print("✓ Modelo listo")

✓ Modelo listo


In [6]:
# ══════════════════════════════════════
#  ZERO-SHOT PROMPTING - Vulcanizadora
# ══════════════════════════════════════

def chat_zero_shot(pregunta):
    prompt = f"""Eres un profesional experto en vulcanizado y neumáticos.

Tarea: Responde la consulta del cliente de forma clara y profesional.

Formato de respuesta:
- Diagnóstico: [qué tiene el neumático]
- Recomendación: [parchar o cambiar, con razón]
- Seguridad: [¿es urgente?]

Consulta del cliente: "{pregunta}"
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"🙋 Cliente      : {pregunta}")
    print(f"🔧 Vulcanizador : {response.content}")
    print("-" * 50)

# Pruebas
chat_zero_shot("Tengo un neumático con alambres salidos y forma de huevo, ¿lo parcho o lo cambio?")
chat_zero_shot("¿Puedo seguir manejando con un clavo en el neumático?")

🙋 Cliente      : Tengo un neumático con alambres salidos y forma de huevo, ¿lo parcho o lo cambio?
🔧 Vulcanizador : - Diagnóstico: Su neumático presenta alambres expuestos y deformación ("forma de huevo"). Esto indica daño estructural severo en la carcasa y los cinturones internos del neumático.

- Recomendación: Debe cambiar el neumático de inmediato. No es posible ni seguro repararlo con un parche, ya que la integridad del neumático está comprometida y no garantiza un funcionamiento seguro.

- Seguridad: Es urgente. Circular con un neumático en estas condiciones aumenta el riesgo de reventón y pérdida de control del vehículo. Le recomiendo no utilizar el vehículo hasta reemplazar el neumático afectado.
--------------------------------------------------
🙋 Cliente      : ¿Puedo seguir manejando con un clavo en el neumático?
🔧 Vulcanizador : - Diagnóstico: Su neumático tiene un objeto punzante (clavo) incrustado, lo que puede causar pérdida de aire, daño interno o incluso un reventón si

In [7]:
# ══════════════════════════════════════
#  FEW-SHOT PROMPTING - Vulcanizadora
# ══════════════════════════════════════

def chat_few_shot(pregunta):
    prompt = f"""Eres un profesional experto en vulcanizado y neumáticos.
Responde siguiendo exactamente el estilo de estos ejemplos:

---
Cliente: "Tengo un clavo en la banda de rodadura del neumático"
Vulcanizador:
- Diagnóstico: Pinchazo en zona reparable (banda central)
- Recomendación: ✅ SE PUEDE PARCHAR - parche interno vulcanizado
- Seguridad: No es urgente, pero no demores más de 1 día

---
Cliente: "Mi neumático tiene una burbuja en el costado"
Vulcanizador:
- Diagnóstico: Hernia lateral, la carcasa interna está rota
- Recomendación: ❌ DEBE CAMBIARSE - no tiene reparación posible
- Seguridad: ⚠️ URGENTE, puede reventar al manejar

---
Cliente: "El neumático tiene un corte en el flanco de 3 cm"
Vulcanizador:
- Diagnóstico: Corte en zona no reparable (flanco lateral)
- Recomendación: ❌ DEBE CAMBIARSE - los flancos no se parchean
- Seguridad: ⚠️ URGENTE, riesgo de desinflado repentino

---
Cliente: "{pregunta}"
Vulcanizador:"""

    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"🙋 Cliente      : {pregunta}")
    print(f"🔧 Vulcanizador : {response.content}")
    print("-" * 50)

# Pruebas
chat_few_shot("Tengo un neumático con alambres salidos y forma de huevo, ¿lo parcho o lo cambio?")
chat_few_shot("¿Puedo seguir manejando con un clavo en el neumático?")

🙋 Cliente      : Tengo un neumático con alambres salidos y forma de huevo, ¿lo parcho o lo cambio?
🔧 Vulcanizador : - Diagnóstico: Desgaste severo, deformación ("huevo") y exposición de alambres
- Recomendación: ❌ DEBE CAMBIARSE - no es reparable, el daño es estructural
- Seguridad: ⚠️ URGENTE, riesgo alto de reventón y pérdida de control
--------------------------------------------------
🙋 Cliente      : ¿Puedo seguir manejando con un clavo en el neumático?
🔧 Vulcanizador : - Diagnóstico: Pinchazo por objeto extraño (clavo) en el neumático  
- Recomendación: ✅ SE PUEDE PARCHAR - si está en la banda de rodadura, parche interno vulcanizado  
- Seguridad: No es urgente si no pierde aire, pero no demores más de 1 día. Si pierde aire, ⚠️ URGENTE, riesgo de desinflado repentino
--------------------------------------------------
